## Setup Environment

Import required libraries and configure the OpenAI API key by reading it from 1Password using the CLI and your personal account.

**Prerequisites:**
- [1Password CLI](https://developer.1password.com/docs/cli/get-started/) installed (`brew install 1password-cli`).
- 1Password desktop app open with **Settings → Developer → Integrate with 1Password CLI** enabled.
- Create a `.env` file from `.env.example` and set `OP_SECRET_REF` to your `op://vault/item/field` reference.

In [ ]:
import os
import subprocess
import openai
from dotenv import load_dotenv

# Load OP_SECRET_REF from .env (this is just a path reference, not a real secret)
load_dotenv()

secret_ref = os.environ.get("OP_SECRET_REF")
if not secret_ref:
    raise RuntimeError("OP_SECRET_REF is not set. Add it to your .env file.")

# Clear any existing 1Password CLI session (for testing this E2E)
subprocess.run(["op", "signout"])

# Requires biometric unlock: 1Password app → Settings → Developer → "Integrate with 1Password CLI"
result = subprocess.run(
    ["op", "read", secret_ref],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(f"1Password CLI error:\n{result.stderr.strip()}")

openai.api_key = result.stdout.strip()
print("OpenAI client configured via 1Password CLI.")

OpenAI client configured via 1Password CLI.


## Initialize ChromaDB Client

Create a persistent ChromaDB client that stores data on disk under `./chroma_data`.

In [2]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_data")
print(f"ChromaDB client initialized with persistent storage at ./chroma_data")

ChromaDB client initialized with persistent storage at ./chroma_data


## Get or Create Collection

Retrieve an existing ChromaDB collection by name, or create it if it doesn't exist yet.

In [3]:
collection = client.get_or_create_collection(name="my_collection")
print(f"Collection '{collection.name}' is ready. Count: {collection.count()}")

Collection 'my_collection' is ready. Count: 0


## References

Official documentation and resources supporting the approach used in this notebook.

### 1Password Secrets Management
- [1Password CLI — Get Started](https://developer.1password.com/docs/cli/get-started/) — Install and authenticate the `op` CLI with your personal account.
- [Secret References](https://developer.1password.com/docs/cli/secret-references/) — Syntax for `op://vault/item/field` references used with `op read`.
- [op read](https://developer.1password.com/docs/cli/reference/commands/read/) — CLI command used to retrieve a secret value directly.

### Environment & Configuration
- [python-dotenv Documentation](https://saurabh-kumar.com/python-dotenv/) — Load variables from a `.env` file into `os.environ` at runtime.

### ChromaDB
- [ChromaDB Documentation](https://docs.trychroma.com/) — Official docs covering collections, embeddings, querying, and persistent storage.
- [ChromaDB GitHub](https://github.com/chroma-core/chroma) — Source code and issue tracker.
- [Persistent Client Guide](https://docs.trychroma.com/docs/run-chroma/clients#persistent-client) — How `PersistentClient` works and stores data on disk.

### OpenAI Embeddings
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings) — Overview of text embedding models and best practices.
- [OpenAI Python SDK](https://github.com/openai/openai-python) — Official Python client library.

### Vector Databases & Retrieval-Augmented Generation (RAG)
- [What is a Vector Database? (Pinecone)](https://www.pinecone.io/learn/vector-database/) — Conceptual explanation of vector databases and similarity search.
- [Retrieval-Augmented Generation (RAG) — AWS](https://aws.amazon.com/what-is/retrieval-augmented-generation/) — Explains the RAG pattern that underpins this local ChromaDB approach.